# Module 1: Power Query Formula Language (M)
### Week 10 | June 8, 2026

Last week M was the code running quietly under all my Power Query button-clicks. This week I stop letting the GUI write it for me and start reading and writing **M** by hand. M is still the **Power Query** half of the pipeline (it cleans/shapes data *before* it enters the model, with no awareness of report filters) — DAX is still the separate "after the model" language.

This module covers two sections: introducing M (the `let`/`in` structure, naming, formatting, and how steps flow), and the M language lexicon (values, types, operators, and evaluation).

## By End of Module 1 I Should Be Able To:
- [ ] Explain what M is and why it's worth learning over the GUI alone
- [ ] Read a `let`/`in` query and identify the steps and the final output
- [ ] Name steps correctly, including the `#"..."` rule for spaces
- [ ] Explain how M's evaluation engine orders steps by dependency
- [ ] Describe why out-of-order steps work but break the Applied Steps panel
- [ ] Compare M to Python at a high level (purpose, where it runs, style)

<h2 style="color:steelblue">Section 1.A: Introducing M</h2>

### 1.A.1 M Language

M (**Power Query Formula Language**) is the language behind Power BI's Power Query Editor. The big idea: learning M lets me do advanced transformations the GUI handles poorly or can't do at all.

- A lot of work that's *possible* through the GUI is **faster** to just write in M.
- **Cut and paste M** to reuse transformation logic across queries instead of re-clicking everything. ⭐
- **Case-sensitive** ⚠️ — `Source` and `source` are different things.

Microsoft describes M as a "mostly pure, higher-order, dynamically typed, partially lazy, functional language," similar to **F#**. It's called a **mashup language** because it's built to *combine* data from different sources.

> 💡 Same case-sensitivity instinct as pandas (`df` ≠ `Df`). Different from SQL, where keywords usually aren't case-sensitive.

### 1.A.2 Programming for Non-Programmers

M was designed with **Excel power users** in mind, not programmers, so it feels different from other query languages.

- M breaks a transformation into **individual named steps** — easier to think through a long task list one piece at a time.
- M's **evaluation engine** works out each expression's dependencies and decides the compute order itself.
- The payoff: I focus on **what** needs doing, not **when**. The engine handles ordering.

> 💡 This is exactly like a **spreadsheet**. In Excel, `C1 = A1+B1` and Excel knows to compute A1 and B1 first — I never specify the order. M works the same way.

Reasons to learn it: faster data prep, greater flexibility than the GUI, easy reuse via copy/paste, and a real understanding of what Power Query is doing under the hood.

### What M Is Used For

M is the language **used by** Power Query — the "clean before it enters the model" stage from Week 9. So M does the pre-model work:

- Import data
- Clean data
- Transform data
- Merge tables
- Create custom columns
- Build **reusable** data preparation workflows

> 💡 Pandas equivalents: `pd.read_csv()` (import), `.dropna()` (clean), `.rename()`/`.astype()` (transform), `.merge()` (merge tables). Same jobs, different language.

### 1.A.3 Let Expressions

A `let` expression is the **first thing** most people notice in M — the skeleton every query is built on. An M query has **two main sections**:

| Keyword | Job |
|---|---|
| `let` | Defines variables and transformation steps |
| `in` | Specifies the final result returned |

The basic framework: `let` (steps go here) ... `in` (final output goes here).

> 🔁 Tip: in the Advanced Editor, use the **Display Options** drop-down to turn on **word-wrap** so long queries stay readable.

### The Structure — The Pattern Every Query Follows

Every Power Query query follows this pattern: `let` → `Step1 = ...,` `Step2 = ...,` `Step3 = ...` → `in` → `Step3`.

- `let` defines the **named expressions** (the steps).
- `in` returns the **final result** — usually just the name of the last step.

⚠️ **Two syntax gotchas:**
- Comma after **every** step EXCEPT the last one before `in`.
- `in` returns **one** named step, not a second list.

> 💡 Pandas shape: the `let` block is my run of assignments (`Step1 = ...`, `Step2 = Step1...`, `Step3 = Step2...`) and `in Step3` is the `return Step3` at the end.

### The Structure — Expression Flow

Data flows top to bottom, each step feeding the next:

`Source Data` → `Step 1: Import` → `Step 2: Clean` → `Step 3: Transform` → `Final Output (in)`

Each step takes the previous step's result as its starting point — the **method-chaining** mental model from pandas, just written as named lines instead of dots.

### Worked Example — Steps as Variables

A query from the Advanced Editor. Comment block first (`/* ... */`), then the `let`/`in`:

    let
        Number1 = 100,
        Number2 = 50,
        Total = Number1 + Number2
    in
        Total

The point: **M stores each transformation as a variable**, and a step can reference earlier steps — `Total` uses both `Number1` and `Number2`. The editor confirms "No syntax errors detected." Note the commas after the first two steps and none after `Total`. ✅

> 💡 Pandas: `Number1 = 100`, `Number2 = 50`, `Total = Number1 + Number2`, then `return Total`. Identical thinking.

### Worked Example — Create a Simple Query (`#table`)

Building a literal table from scratch with the `#table` constructor:

    let
        Source = #table(
            {"Product", "Sales"},
            { {"Laptop", 1200}, {"Mouse", 300}, {"Monitor", 800} }
        )
    in
        Source

How `#table` reads: first the **list of column names** `{"Product", "Sales"}`, then a **list of rows**, where each row is itself a list `{"Laptop", 1200}`. So it's a list-of-lists for the data. ⚠️ Those `{ }` braces are M's list syntax (more on that in 1.B).

> 💡 Pandas: `pd.DataFrame({"Product": [...], "Sales": [...]})` — same idea, hand-building a small table inline.

### 1.A.4 Expression Naming

Named expressions show up **as steps in the designer** (the Applied Steps panel), so people give them readable names with spaces ("Removed Columns", "Filtered Rows").

The rule ⚠️: **if a step name has a space, wrap it in quotes and prefix with `#`** → `#"Removed Columns"`. A no-space name like `NameOne` doesn't need it.

Example from the guide:

    let
        NameOne = "...does not contain spaces...",
        #"Name Two" = "...does contain a space..."
    in
        #"Name Two"

> 💡 This is why the auto-generated M from Week 9 was full of `#"Removed Other Columns"` — the GUI names steps with spaces, so it *has* to use the `#"..."` syntax.

### 1.A.5 Formatting Expressions

White space (spaces, tabs, line breaks) can be added freely — **like SQL**, where indentation doesn't change behavior but makes things readable.

- Good formatting makes a query much easier to read and write.
- Tool **powerqueryformatter.com** cleans queries up better than Power Query Editor does on its own. 🔁

> 💡 Same instinct as pandas/SQL: the engine ignores my line breaks, but future-me doesn't.

### 1.A.6 Let Expression Sets

A `let` expression lets a **set** of expressions each get a name. The rules in one place:

- Each expression can **use the result of other expressions**.
- **Comma** after each expression, except the last before `in`.
- The `in` statement finishes the set and declares the final result — usually just the name of one expression above.

This is the formal version of the structure pattern: a named, comma-separated list of steps, capped by an `in` that picks the output.


### 1.A.7 Let Expression Flow (In Order)

The clean case: **each step depends only on steps defined before it.** Step2 uses Step1, Step3 uses Step2, reading naturally top to bottom.

This is the way I should always aim to write it — it matches how a human reads and how the Applied Steps panel expects things to run.

### 1.A.8 Let Expression Flow: Out of Order

The surprising part — if I reverse the steps so one references something defined *below* it:

- ✅ It **still works**. M resolves by **dependency**, not line position (the spreadsheet idea again — Excel doesn't care if the cell it needs is above or below).
- ⚠️ BUT the **Applied Steps panel in the GUI breaks**, because the designer expects steps in order.

**The rule:** always order steps so each depends only on **previous** steps. Just because out-of-order *can* work doesn't mean I should write it that way.

### M Functions

M functions are part of the Power Query M language (functional, case-sensitive, built for data prep in Power BI, Excel, and other Microsoft tools). Each transformation step in Power Query Editor — removing columns, changing text case, adding a calculated column — **corresponds to an M function**. Functions are organized into categories by data type or operation.

Two kinds:

| Type | What It Means |
|---|---|
| **Built-in** | Comes with M's standard library, ready to use without defining |
| **User-defined** | A function I write myself for custom logic |

There are **over 700 built-in functions**, and Microsoft adds more with each yearly update. Best reference is Microsoft's official M function reference (docs.microsoft.com/powerquery-m).

> 💡 Like pandas: tons of built-in methods (`.dropna()`, `.merge()`) plus the ability to write my own functions when needed.

### Built-in Function Categories

The ten built-in categories, each grouped by what kind of data or operation it handles:

| # | Category | What It Does |
|---|---|---|
| 1 | **Table** | Create, query, manipulate tables — joins, filtering, adding columns |
| 2 | **Text** | Transform/combine text — formatting, splitting, extracting substrings |
| 3 | **Number** | Arithmetic, rounding, comparisons on numeric values |
| 4 | **Date and Time** | Extract, manipulate, combine date/time/datetime values |
| 5 | **List** | Create/manipulate lists — sorting, filtering, aggregating |
| 6 | **Record** | Create, access, modify structured records |
| 7 | **Logical** | Handle true/false conditions and conditional logic |
| 8 | **Type** | Manage data types and conversions |
| 9 | **Error Handling** | Detect and manage errors in queries |
| 10 | **Binary and URI** | Work with binary data and URLs — encoding/decoding |

> 💡 The categories I'll touch most in Northwind/Contoso work: **Table** (the structural stuff), **Text** (cleaning string columns), and **Date and Time** (anything time-related).

### Python vs Power BI M Language

Where M fits relative to the Python I already know:

| Feature | Python | Power BI M Language |
|---|---|---|
| **Primary Purpose** | General-purpose: software dev, data analysis, automation, AI, ML | Specialized: extraction, transformation, loading (ETL) in Power Query |
| **Where It Runs** | Jupyter, VS Code, PyCharm, command-line | Power Query Editor in Power BI, Excel, Microsoft Fabric |
| **Data Analysis** | Advanced analytics/stats/ML via pandas, NumPy, scikit-learn | Cleaning, reshaping, prepping data *before* it loads into Power BI |
| **Programming Style** | Procedural, object-oriented, and functional | Functional, based on `let` expressions and transformation steps |
| **Typical Use Cases** | Apps, automation, ML models, web | Filtering rows, removing columns, merging tables, changing data types, repeatable ETL |

> 💡 The key takeaway: Python is a *Swiss Army knife*, M is a *specialized ETL tool*. M's whole job is the data-prep slice that pandas also does — but M does it *before* the model, and that's all it does.

### 🔁 Confusion Corner

**"If M figures out the order itself, why does step order matter?"**

Two layers. The M **engine** resolves by dependency (out-of-order runs fine). The **GUI's Applied Steps panel** reads top-to-bottom and breaks when code isn't in order. I write in dependency order to keep the GUI happy, not because the engine needs it.

**"When do I need `#"..."`?"**

Only when a step name has a **space** (or special character). `RemovedColumns` is fine bare; `#"Removed Columns"` needs the wrapper.

**"Comma or no comma?"**

Comma after **every** step except the **last one before `in`**. This is the #1 starter syntax error. ⚠️

**"Is M's `in` like Python's `in`?"**

No. Python's `in` checks membership (`x in list`). M's `in` just marks the query's final output. Same word, unrelated jobs.

**"Built-in vs user-defined function?"**

Built-in comes free with M's standard library (700+ of them). User-defined is one I write myself. Same split as pandas methods vs my own `def`.

<h2 style="color:steelblue">Section 1.B: M Language Lexicon</h2>

Now the building blocks. 1.A was the skeleton (`let`/`in`); 1.B is the vocabulary — what counts as a value, the types, and the operators for reaching into them.

### 1.B.1 M Expressions and Values

The central construct in M is the **expression**. An expression can be **evaluated** (computed), yielding a single **value**.

- An expression is a **recipe** for evaluation.
- A value is the **result** of evaluation.
- Many values can be written literally as an expression, but a value is *not* itself an expression.

Examples: `SomeValue = 42` (the expression `42` evaluates to the value `42`), and `AnotherValue = 40 + 2` (the expression `40 + 2` also evaluates to `42`).

> 💡 Python parallel: `40 + 2` is an expression; `42` is the value it produces. Same distinction.

### 1.B.2 Types of Values

M values come in two families:

| Family | Meaning | Members |
|---|---|---|
| **Primitive** | Single-part value | Null, Logical (Boolean), Number, Time, Date, DateTime, DateTimeZone, Duration, Text, Binary |
| **Structured** | Built from one or more primitives or other structured values | List, Field, Record, Table, Function, Type |

> 💡 Like pandas: primitives are the scalar dtypes (int, float, bool, str, datetime); structured types are the containers (Series, DataFrame, dict-like records).

### 1.B.3 Primitive Values

A primitive value is a single-part value — a number, date, text, or null. `null` indicates the **absence of any data**.

| Type | Literal example |
|---|---|
| Number | `0`, `1.5`, `2.3e-5`, `3.14`, `0xff` (hex = 255) |
| Text | `"abc"`, `"hello student"` |
| Logical | `true`, `false` |
| Null | `null` (no data) |
| Date | `#date(2023, 12, 31)` |
| DateTime | `#datetime(2023, 12, 31, 12, 0, 0)` |
| Duration | `#duration(15, 35, 0, 0)` |
| Time / DateTimeZone | `#time(09,15,00)`, `#datetimezone(...)` |

⚠️ Time-based values are stored as a single value but written through **special `#` functions** (`#date`, `#time`, etc.) — not plain literals. Numbers allow whole numbers, fractions, scientific notation (`1.0e3`), and hex (`0xff`).

> 💡 The `#date(y, m, d)` constructor is M's version of `datetime.date(y, m, d)` in Python.

### 1.B.4 List Values

A list value is an **ordered sequence** of values. **Curly braces `{ }`** denote the start and end of a list.

- M supports infinite lists, but a literal list must have a **fixed length**.
- A list can mix types: `{42, true, "orange"}` (a number, a logical, a text).
- Or be uniform: `{"red", "orange", "yellow", "green", "blue", "indigo", "violet"}`.

> 💡 This is M's version of a Python **list** — `[42, True, "orange"]` — just with `{ }` instead of `[ ]`. (Watch out: in M, `[ ]` means something totally different — that's records. More below.)

### Worked Example — List Index (`{ }`)

From the Advanced Editor:

    let
        Colors = {"Red", "Blue", "Green", "Yellow"},
        Result = Colors{2}
    in
        Result

`Colors{2}` returns **"Green"** — because list indexing is **zero-based** (Red=0, Blue=1, Green=2). ✅

> 💡 Exactly like Python: `colors[2]` is the third item. M just uses `{ }` instead of `[ ]` for the index. ⚠️ Don't mix them up.

### 1.B.5 Fields and Record Values

A **field** is a name-value pair, where the name is text and is unique within its record. A field must belong to a record. A **record** is a set of fields.

- Record syntax uses **square brackets `[ ]`**.
- Field names are written **without quotes** (called *identifiers*).

Example — a record with five fields:

    [ A = 2, B = 3, C = 5, D = 7, E = 11 ]

> 💡 A record is basically a Python **dict** — `{"A": 2, "B": 3, ...}` — or one row of data. Key difference: M uses `[ ]` and skips the quotes on names.

### 1.B.6 Table Values

A table is a set of values organized into **columns and rows**, with columns identified by name (headers).

⚠️ There is **no literal syntax** for creating a table directly — instead you use standard functions like `#table` (or `Table.FromRecords`) to build one from lists or records.

Example with `#table`:

    #table(
        {"Product", "Description"},
        { {"Super Soaker", "..."}, {"Pog", "..."}, {"Gameboy", "..."} }
    )

> 💡 The Table is M's **DataFrame** — the main structure everything else feeds into. It's literally what Power Query hands to the model.

### 1.B.7–1.B.8 Function Values (and Functions Are Values)

> Merged: this cell covers two guide subsections — 1.B.7 (Function Values) and 1.B.8 (Functions are Values).

A **function** is a value that, when invoked with arguments, produces a new value. Written as: **parameters in parentheses** → the **such-that symbol `=>`** → the **expression** defining it.

Example — average of two numbers: `(x, y) => (x + y) / 2`.

Because a function is itself a value, it can live inside a record and be called by other fields:

    [
        Add = (x, y) => x + y,
        OnePlusOne = Add(1, 1),
        TwoPlusOne = Add(2, 1)
    ]

M also ships a **standard library** of functions available without defining them (e.g. `Number.E`, `Text.PositionOf("Hello", "llo")` → `2`).

> 💡 `(x, y) => (x + y) / 2` is M's **lambda** — same as Python's `lambda x, y: (x + y) / 2`. And "functions are values" is the same idea as Python treating functions as first-class objects.

### Worked Example — Defining and Calling a Function

    let
        MyFunction = (a, b) => (a + b) / 2
    in
        MyFunction(2, 4)

Defines `MyFunction` with `(a, b) =>`, then **invokes** it on `2` and `4`, returning their average (`3`). ✅

> 💡 `def my_function(a, b): return (a + b) / 2`, then `my_function(2, 4)`. Identical.

### 1.B.9 M Function Library Reference

The M standard library ships **700+ built-in functions**, and Microsoft adds more with each yearly update. Best reference is Microsoft's official M function reference (docs.microsoft.com/powerquery-m).

(The ten function categories and built-in vs user-defined are covered in the **M Functions** cells above.)

### 1.B.10–1.B.11 Expression Evaluation (and a Caution)

M's evaluation model is **like a spreadsheet**: parts of an expression reference other parts by name, and M **automatically determines the calculation order** by building a **dependency tree** behind the scenes.

⚠️ **The caution:** dependency-based evaluation is powerful, but it has side effects. If a function modifies **external state** or returns a **different value each time**, the execution order suddenly matters and you'd have to reverse-engineer the dependencies to follow what's happening.

**Fix:** favor **deterministic** functions — same inputs always give the same output.

| Deterministic ✅ | Non-deterministic ⚠️ |
|---|---|
| Removing a column | Retrieving data from a database |
| Adding two values | Subtracting a birthdate from the *current* time |

> 💡 Same idea as writing **pure functions** in Python — no side effects, predictable output. Easier to reason about and debug.

### 1.B.12 Nesting and the Lookup Operator `[ ]`

Records can be **nested** inside other records. Use the **lookup operator `[ ]`** to access a field by name.

Example — a `Sales` record nested inside, with `Total` reaching into it:

    [
        Sales = [ FirstHalf = 1000, SecondHalf = 1100 ],
        Total = Sales[FirstHalf] + Sales[SecondHalf]
    ]

So `Sales[FirstHalf]` pulls the `FirstHalf` field out of the `Sales` record.

> 💡 `Sales[FirstHalf]` is like `sales["FirstHalf"]` on a Python dict. ⚠️ Reminder: `[ ]` = look up a field **by name**; `{ }` = grab a list item **by position**. Two different operators.

### 1.B.13 Positional Index Operator `{ }`

Records can sit inside lists. Use the **positional index operator `{ }`** to grab a list item by its **zero-based** numeric index.

You can chain it with `[ ]` to drill in — e.g. with a list of yearly-sales records:

`TotalSales = Sales{0}[Total] + Sales{1}[Total]`

This reads as: take list item `0`, then its `Total` field; add list item `1`'s `Total`.

> 💡 `Sales{0}[Total]` ≈ `sales[0]["Total"]` in Python — index into the list, then key into the dict. Same chaining, M just splits the syntax: `{ }` for position, `[ ]` for name.

### 1.B.14 Let Expressions Compared with Lists

A `let` expression can **replace the positional operator** by giving each record its own **name** instead of indexing by position.

So instead of `Sales{0}[Total] + Sales{1}[Total]`, you write:

    let
        Sales2017 = [ Year = 2017, FirstHalf = 2000, SecondHalf = 1100, Total = FirstHalf + SecondHalf ],
        Sales2018 = [ Year = 2018, FirstHalf = 2200, SecondHalf = 1300, Total = FirstHalf + SecondHalf ]
    in
        Sales2017[Total] + Sales2018[Total]

Both return the same result; the `let` version is more readable because the names say what each thing *is*.

It's also acceptable (though uncommon) to **nest `let` expressions** — a little like using **subqueries in SQL**.

> 💡 Named steps > magic index numbers. Same reason I'd name a pandas variable instead of relying on `df.iloc[0]` everywhere — readability and fewer silent bugs.

### 1.B.15 Lazy vs Eager Evaluation

| Evaluation | Applies to | Behavior |
|---|---|---|
| **Lazy** | Lists, Tables, Record members, Let expressions | Computed **only when accessed** — may *never* be evaluated at all |
| **Eager** | All other expressions | Evaluated **immediately** as encountered |

> 💡 Lazy evaluation is like a Python **generator** — nothing computes until you actually pull a value out. It's part of why M is "partially lazy" (from that 1.A.1 definition).

### 1.B.16 If Expression

`if` conditionally picks one of two expressions based on a test. Form: `if <condition> then <value> else <value>`.

Example: `if CustomerAge < 18 then "RED" else "GREEN"`.

Worked demo from class:

    let
        Score = 85,
        Result = if Score >= 70 then "Pass" else "Fail"
    in
        Result

Returns "Pass". ✅

> 💡 M's `if/then/else` is an **expression** that returns a value, so it's closer to Python's ternary (`"Pass" if score >= 70 else "Fail"`) than to a multi-line `if` block. ⚠️ `else` is required.

### 1.B.17 Each Keyword

`each` is shorthand for "do this for **every row/input**." Inside the function after `each`, the **underscore `_`** represents the current record. If you only need a field name, the `_` is **implied** — so `each _[Price]` and `each [Price]` mean the same thing.

Worked demo — adding a tax column:

    let
        Source = Table.FromRecords({
            [Product="Laptop", Price=1000],
            [Product="Tablet", Price=500],
            [Product="Phone", Price=800]
        }),
        AddTax = Table.AddColumn(Source, "PriceWithTax", each [Price] * 1.08)
    in
        AddTax

`each [Price] * 1.08` runs once per row, building a new `PriceWithTax` column.

> 💡 `each` is M's row-wise apply. `Table.AddColumn(Source, "PriceWithTax", each [Price] * 1.08)` ≈ `df["PriceWithTax"] = df["Price"] * 1.08`, and `each _` ≈ the `row` in `df.apply(lambda row: ..., axis=1)`. The `_` is the current row.

### 1.B.18–1.B.19 Comments and Operator Order

**Comments** explain a query and are **not interpreted as code**. M uses `//` for single-line and `/* ... */` for block comments (that's the author/date header on every demo).

**Operator Order of Evaluation:** like algebra, expressions evaluate based on **operator precedence**, *not* strictly left-to-right. (Full precedence table in Microsoft's docs.)

> 💡 Same as Python/SQL — `2 + 3 * 4` is `14`, not `20`, because `*` binds tighter than `+`. When unsure, use parentheses.

### How M Executes Code (Summary)

Pulling the execution model together:

**Expression Evaluation**
- Uses **dependency-based** evaluation
- **Similar to spreadsheet** calculations
- **Automatically determines** execution order

**Additional Concepts**
- **Lazy evaluation:** calculated only when needed
- **Comments** improve readability
- **Operators** follow a defined order of evaluation

**Best Practice**
- Write **deterministic** functions whenever possible
- Use **comments and formatting** for maintainability

### 🔁 Confusion Corner (1.B)

**"`{ }` vs `[ ]` — which is which?!"** ⚠️ The single biggest M mix-up:
- `{ }` = **list**, accessed by **position** (zero-based): `Colors{2}` → 3rd item.
- `[ ]` = **record**, accessed by **field name**: `Sales[FirstHalf]` → that field.
Chain them to drill in: `Sales{0}[Total]` = list item 0, then its `Total` field. (In Python both use `[ ]`; M splits them. That's the trap.)

**"Expression vs value?"** Expression = the recipe (`40 + 2`). Value = the result (`42`).

**"Why care if a function is deterministic?"** Because M decides execution order by dependency. A function that changes each run (DB pull, "current time") makes order unpredictable. Deterministic = same input, same output = safe.

**"Is `each [Price]` missing something?"** No — the `_` is implied. `each [Price]` and `each _[Price]` are identical. The `_` is the current row.

**"`#date(...)` — why the `#`?"** Date/time values aren't plain literals; they're built with `#` constructor functions (`#date`, `#time`, `#datetime`, `#duration`).

### ⭐ When Would I Use This? (Module 1)

- **Reading my Week 9 M:** open Advanced Editor on any Northwind query — I can now read the `let`/`in`, the named steps, the `#"..."` names, and the functions on each line.
- **Adding a calculated column without the GUI:** `Table.AddColumn(Source, "NewCol", each [A] * [B])` — the M behind "Add Column."
- **Conditional flags:** `if [Status] = "Active" then 1 else 0` to bucket rows.
- **Reaching into nested data:** APIs and JSON come in as nested records/lists — `Source{0}[fieldName]` drills in.
- **Building test data:** `#table` or `Table.FromRecords` to hand-make a sample table when I don't want to touch a live source.
- **Writing reusable functions:** `(x, y) => ...` for logic I repeat across queries.
- **Debugging weird results:** check for non-deterministic steps or `{ }`/`[ ]` mix-ups.

### Key Takeaways — Module 1 🎯

Big picture: **M is the code under Power Query**, and this module taught me to read and write it instead of just clicking buttons. Every query is a `let ... in` sandwich — named steps with commas (none after the last), and `in` points at the final answer. M figures out the order itself by dependencies, exactly like a spreadsheet, so I focus on *what*, not *when*.

The vocabulary (1.B) is really just **values and how to reach into them**. Primitives are the simple stuff (number, text, logical, null, dates via `#date(...)`). Structured types are the containers: **lists** `{ }` (ordered, zero-indexed, like Python lists), **records** `[ ]` (name-value, like dicts), and **tables** (the DataFrame everything feeds). The one thing I cannot afford to mix up: **`{ }` is position, `[ ]` is name** — and I chain them (`Sales{0}[Total]`) to drill into nested data.

Functions are values too, written `(x, y) => ...` (M's lambda), and there are 700+ built-in ones across ten categories. `if/then/else` is an expression that returns a value (always needs `else`), and `each` is row-wise apply where `_` is the current row — `each [Price] * 1.08` is just `df["Price"] * 1.08`.

Execution model to remember: **dependency-based, lazy, deterministic-when-possible.** Write clean functions, comment with `//` or `/* */`, format for readability, and let operator precedence work like it does in algebra. That's Module 1 — I can now open any query and actually *read* it. ✅